# Objective 2 Task 3 — Daily Demand Model Integration

This notebook integrates the Task 3 demand-modelling work into one clean workflow.

It keeps the comparison trail from the source notebooks rather than jumping straight to the final model:

- `merge.ipynb`: source-data merge and QA checks.
- `EDA.ipynb`: exploratory checks used to justify weather and calendar features.
- `gam_rf_pri_aligned.ipynb`: aligned GAM and random-forest comparison.
- `gam_year_smoothing_additional_feature_testing.ipynb`: extra temporal feature tests for GAMs.
- `Objective 2 Task 3.ipynb`: colleague's linear-regression and XGBoost workflow, including the final expanded XGBoost model selection.

The final selected modelling family is XGBoost, but the notebook deliberately retains the earlier model-family comparison.

## Modelling contract

- Boundary: Great Britain electricity demand.
- Historic demand target: `nd_daily_mwh`.
- Primary demand definition: NESO National Demand (`ND`).
- Daily model purpose: learn climate-sensitive daily demand from historic demand, weather, and calendar features.
- Future use: predict future daily climate response before FES annual anchoring and hourly disaggregation.
- Final Objective 2 output downstream: `demand_future_hourly_2030_2045`.

Generated Parquet/model artefacts should stay local unless they are deliberately committed as small validation evidence.

In [ ]:
from __future__ import annotations

import json
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from IPython.display import display
except ImportError:  # pragma: no cover
    display = print

try:
    from pygam import LinearGAM, s, f
    HAS_PYGAM = True
except ImportError:
    HAS_PYGAM = False

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore", category=FutureWarning)

## 1. Paths and run settings

Set these paths once. The notebook can either:

1. load an already merged `demand_model_train_ready.parquet`, or  
2. merge demand and weather feature Parquets on `date`.

For the repo version, keep large Parquet outputs out of Git.

In [ ]:
# Assumes the notebook is run from the repository root or a subfolder of it.
def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "config").exists() and (candidate / "data").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()

# Adjust this if your local files are in a staging folder during integration.
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "objective2_demand"
LOCAL_STAGING_DIR = PROJECT_ROOT / "data" / "interim" / "objective2_demand"

# The loader checks these locations in order.
DATA_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_STAGING_DIR.mkdir(parents=True, exist_ok=True)

DATA_SEARCH_DIRS = [
    DATA_DIR,
    LOCAL_STAGING_DIR,
    Path.cwd(),
]

TRAIN_READY_FILENAME = "demand_model_train_ready.parquet"
DEMAND_DAILY_FILENAME = "demand_model_training_data.parquet"
WEATHER_FEATURES_FILENAME = "weather_hist_features_daily.parquet"

# Local-only outputs. Keep generated artefacts out of Git unless deliberately using tiny CSV/Markdown as validation evidence.
LOCAL_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "objective2_demand" / "task3_demand_model"
VALIDATION_DIR = PROJECT_ROOT / "docs" / "validation" / "objective2_demand"

WRITE_LOCAL_OUTPUTS = False
WRITE_VALIDATION_EVIDENCE = False

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_SEARCH_DIRS:")
for p in DATA_SEARCH_DIRS:
    print(" -", p)


## 2. Load and merge model-ready daily data

This replaces duplicated merge logic across the old notebooks. It prefers the pre-merged training file if present; otherwise it performs the explicit `demand + weather` merge used in the colleague notebook.

In [ ]:
def first_existing_file(filename: str, search_dirs: Iterable[Path]) -> Path | None:
    for directory in search_dirs:
        candidate = directory / filename
        if candidate.exists():
            return candidate
    return None


def load_task3_model_data(search_dirs: Iterable[Path]) -> pd.DataFrame:
    train_ready_path = first_existing_file(TRAIN_READY_FILENAME, search_dirs)
    if train_ready_path is not None:
        print(f"Loading pre-merged model data: {train_ready_path}")
        df = pd.read_parquet(train_ready_path)
        df["source_merge_mode"] = "pre_merged"
        return df

    demand_path = first_existing_file(DEMAND_DAILY_FILENAME, search_dirs)
    weather_path = first_existing_file(WEATHER_FEATURES_FILENAME, search_dirs)

    if demand_path is None or weather_path is None:
        searched = "\n".join(str(p) for p in search_dirs)
        raise FileNotFoundError(
            "Could not find either a pre-merged training file or the explicit demand/weather inputs.\n"
            f"Expected either `{TRAIN_READY_FILENAME}` or both "
            f"`{DEMAND_DAILY_FILENAME}` and `{WEATHER_FEATURES_FILENAME}`.\n"
            f"Searched:\n{searched}"
        )

    print(f"Loading demand data:  {demand_path}")
    print(f"Loading weather data: {weather_path}")

    demand_df = pd.read_parquet(demand_path).copy()
    weather_df = pd.read_parquet(weather_path).copy()

    demand_df["date"] = pd.to_datetime(demand_df["date"])
    weather_df["date"] = pd.to_datetime(weather_df["date"])

    model_df = demand_df.merge(weather_df, on="date", how="inner")
    model_df["source_merge_mode"] = "demand_weather_merge"
    return model_df


model_df = load_task3_model_data(DATA_SEARCH_DIRS)
model_df["date"] = pd.to_datetime(model_df["date"])
model_df = model_df.sort_values("date").reset_index(drop=True)

print("Shape:", model_df.shape)
print("Date range:", model_df["date"].min(), "to", model_df["date"].max())
display(model_df.head())

## 3. Schema and QA checks

These checks retain the core merge QA from `merge.ipynb` and the required-column checks from `Objective 2 Task 3.ipynb`.

In [ ]:
REQUIRED_COLUMNS = [
    "date",
    "year",
    "month",
    "day_of_week",
    "is_weekend",
    "is_holiday_eng_wales",
    "is_holiday_scotland",
    "is_holiday_gb_any",
    "nd_daily_mwh",
    "tasmin_gb_c",
    "tasmax_gb_c",
    "tmean_gb_c",
    "hdd",
    "cdd",
]

missing_required = [c for c in REQUIRED_COLUMNS if c not in model_df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

qa_summary = {
    "rows": len(model_df),
    "date_min": model_df["date"].min(),
    "date_max": model_df["date"].max(),
    "duplicate_dates": int(model_df["date"].duplicated().sum()),
    "missing_target_rows": int(model_df["nd_daily_mwh"].isna().sum()),
}

expected_dates = pd.date_range("2010-01-01", "2024-12-31", freq="D")
observed_dates = pd.Index(model_df["date"].dt.normalize())
missing_dates = expected_dates.difference(observed_dates)
extra_dates = observed_dates.difference(expected_dates)

qa_summary["expected_2010_2024_missing_dates"] = len(missing_dates)
qa_summary["extra_dates_outside_2010_2024"] = len(extra_dates)

print("QA summary:")
for key, value in qa_summary.items():
    print(f"{key}: {value}")

print("\nMissing values by required column:")
display(model_df[REQUIRED_COLUMNS].isna().sum().sort_values(ascending=False).to_frame("missing_count"))

if qa_summary["duplicate_dates"] != 0:
    raise ValueError("Duplicate dates found. Resolve before modelling.")

if qa_summary["missing_target_rows"] != 0:
    raise ValueError("Missing target values found. Resolve before modelling.")

## 4. Feature engineering

The first-pass comparable models used `year`, `month`, `day_of_week`, holiday, `hdd`, and `cdd`.

The expanded tests added:

- `time_index`: continuous long-run time trend from 2010-01-01.
- `day_of_year`: within-year seasonal position.

Raw GB temperature variables are kept for EDA but not used in the main model specifications alongside HDD/CDD, to avoid redundant weather representations.

In [ ]:
def prepare_features(df: pd.DataFrame, reference_date: str = "2010-01-01") -> pd.DataFrame:
    out = df.copy()
    out["date"] = pd.to_datetime(out["date"])
    out = out.sort_values("date").reset_index(drop=True)

    # Defensive recomputation from date, so calendar features are internally consistent.
    out["year"] = out["date"].dt.year.astype(int)
    out["month"] = out["date"].dt.month.astype(int)
    out["day_of_week"] = out["date"].dt.dayofweek.astype(int)
    out["day_of_year"] = out["date"].dt.dayofyear.astype(int)
    out["time_index"] = (out["date"] - pd.Timestamp(reference_date)).dt.days.astype(int)

    for col in ["is_weekend", "is_holiday_eng_wales", "is_holiday_scotland", "is_holiday_gb_any"]:
        out[col] = out[col].astype(int)

    numeric_cols = ["nd_daily_mwh", "tasmin_gb_c", "tasmax_gb_c", "tmean_gb_c", "hdd", "cdd"]
    for col in numeric_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    return out


model_df = prepare_features(model_df)

# Save the canonical Task 3 train-ready table for downstream Objective 2 tasks.
TRAIN_READY_PATH = DATA_DIR / TRAIN_READY_FILENAME
model_df.to_parquet(TRAIN_READY_PATH, index=False)
print("Saved train-ready demand model data:", TRAIN_READY_PATH)

display(model_df[[
    "date", "year", "month", "day_of_week", "day_of_year",
    "time_index", "is_holiday_gb_any", "nd_daily_mwh", "tmean_gb_c", "hdd", "cdd"
]].head())


## 5. Concise EDA retained from the source notebooks

The original EDA notebooks had many overlapping plots. This section keeps the plots that directly support feature selection:

- daily demand time series;
- monthly seasonality;
- day-of-week pattern;
- HDD/CDD/temperature relationship;
- compact correlation check.

In [ ]:
eda_cols = ["nd_daily_mwh", "tasmin_gb_c", "tasmax_gb_c", "tmean_gb_c", "hdd", "cdd"]

print("Summary statistics:")
display(model_df[eda_cols].describe().T)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(model_df["date"], model_df["nd_daily_mwh"], linewidth=0.8)
ax.set_title("Daily ND demand over time")
ax.set_xlabel("Date")
ax.set_ylabel("ND daily demand (MWh)")
plt.tight_layout()
plt.show()

monthly_avg = model_df.groupby("month", as_index=False)["nd_daily_mwh"].mean()
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(monthly_avg["month"], monthly_avg["nd_daily_mwh"], marker="o")
ax.set_title("Average daily ND demand by month")
ax.set_xlabel("Month")
ax.set_ylabel("Mean ND daily demand (MWh)")
ax.set_xticks(range(1, 13))
plt.tight_layout()
plt.show()

dow_avg = model_df.groupby("day_of_week", as_index=False)["nd_daily_mwh"].mean()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(dow_avg["day_of_week"], dow_avg["nd_daily_mwh"])
ax.set_title("Average daily ND demand by day of week")
ax.set_xlabel("Day of week: Monday=0")
ax.set_ylabel("Mean ND daily demand (MWh)")
plt.tight_layout()
plt.show()

In [ ]:
for x_col in ["hdd", "cdd", "tmean_gb_c"]:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(model_df[x_col], model_df["nd_daily_mwh"], alpha=0.35, s=10)
    ax.set_title(f"ND daily demand vs {x_col}")
    ax.set_xlabel(x_col)
    ax.set_ylabel("ND daily demand (MWh)")
    plt.tight_layout()
    plt.show()

corr = model_df[eda_cols].corr()
display(corr.round(3))

## 6. Common validation setup

All model families now use one shared evaluation approach:

- development period: all years before 2024;
- final holdout: 2024;
- cross-validation: expanding-window calendar-year validation folds over 2019–2023;
- metrics: MAE, RMSE, WAPE, MBE, and R².

This makes linear regression, GAM, random forest, and XGBoost directly comparable.

In [ ]:
TARGET_COL = "nd_daily_mwh"
TEST_YEAR = 2024

BASE_FEATURES = ["year", "month", "day_of_week", "is_holiday_gb_any", "hdd", "cdd"]
EXPANDED_FEATURES_TIME_INDEX = ["time_index", "month", "day_of_week", "is_holiday_gb_any", "hdd", "cdd"]
EXPANDED_FEATURES_YEAR_DOY = ["year", "day_of_year", "month", "day_of_week", "is_holiday_gb_any", "hdd", "cdd"]
EXPANDED_FEATURES_TIME_INDEX_DOY = ["time_index", "day_of_year", "month", "day_of_week", "is_holiday_gb_any", "hdd", "cdd"]


def regression_metrics(y_true, y_pred) -> dict[str, float]:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "wape": np.abs(y_true - y_pred).sum() / np.abs(y_true).sum(),
        "mbe": (y_pred - y_true).mean(),
        "r2": r2_score(y_true, y_pred),
    }


def make_expanding_year_folds(
    df: pd.DataFrame,
    validation_years: Iterable[int] = range(2019, 2024),
) -> list[dict]:
    folds = []
    for i, val_year in enumerate(validation_years, start=1):
        train_df = df[df["year"] < val_year]
        val_df = df[df["year"] == val_year]

        if train_df.empty or val_df.empty:
            continue

        folds.append({
            "fold": i,
            "train_start": train_df["date"].min(),
            "train_end": train_df["date"].max(),
            "val_start": val_df["date"].min(),
            "val_end": val_df["date"].max(),
            "val_year": val_year,
        })
    return folds


dev_df = model_df[model_df["year"] < TEST_YEAR].copy().reset_index(drop=True)
test_df = model_df[model_df["year"] == TEST_YEAR].copy().reset_index(drop=True)
FOLDS = make_expanding_year_folds(dev_df)

print("Development range:", dev_df["date"].min(), "to", dev_df["date"].max(), dev_df.shape)
print("Test range:", test_df["date"].min(), "to", test_df["date"].max(), test_df.shape)
display(pd.DataFrame(FOLDS))

## 7. Reusable model evaluation functions

In [ ]:
def split_fold_data(df: pd.DataFrame, fold: dict, feature_cols: list[str], target_col: str):
    train_mask = (df["date"] >= fold["train_start"]) & (df["date"] <= fold["train_end"])
    val_mask = (df["date"] >= fold["val_start"]) & (df["date"] <= fold["val_end"])

    train_df = df.loc[train_mask].copy()
    val_df = df.loc[val_mask].copy()

    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_col].to_numpy()
    X_val = val_df[feature_cols].copy()
    y_val = val_df[target_col].to_numpy()

    return train_df, val_df, X_train, y_train, X_val, y_val


def make_tabular_preprocessor(categorical_features: list[str], numeric_features: list[str]) -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", drop=None)),
                ]),
                categorical_features,
            ),
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                ]),
                numeric_features,
            ),
        ],
        remainder="drop",
    )


def evaluate_sklearn_pipeline(
    model_name: str,
    model_factory: Callable[[], Pipeline],
    df: pd.DataFrame,
    dev_df: pd.DataFrame,
    test_df: pd.DataFrame,
    folds: list[dict],
    feature_cols: list[str],
    target_col: str = TARGET_COL,
):
    cv_rows = []
    cv_preds = []

    for fold in folds:
        train_df, val_df, X_train, y_train, X_val, y_val = split_fold_data(
            df=dev_df,
            fold=fold,
            feature_cols=feature_cols,
            target_col=target_col,
        )

        model = model_factory()
        model.fit(X_train, y_train)
        pred_val = model.predict(X_val)
        metrics = regression_metrics(y_val, pred_val)

        cv_rows.append({
            "model_name": model_name,
            "fold": fold["fold"],
            "train_start": fold["train_start"],
            "train_end": fold["train_end"],
            "val_start": fold["val_start"],
            "val_end": fold["val_end"],
            **metrics,
        })

        pred_df = val_df[["date", "year"]].copy()
        pred_df["model_name"] = model_name
        pred_df["split"] = "cv"
        pred_df["fold"] = fold["fold"]
        pred_df["actual_nd_daily_mwh"] = y_val
        pred_df["predicted_nd_daily_mwh"] = pred_val
        pred_df["residual_mwh"] = pred_df["actual_nd_daily_mwh"] - pred_df["predicted_nd_daily_mwh"]
        cv_preds.append(pred_df)

    cv_results = pd.DataFrame(cv_rows)
    cv_predictions = pd.concat(cv_preds, ignore_index=True) if cv_preds else pd.DataFrame()

    final_model = model_factory()
    final_model.fit(dev_df[feature_cols], dev_df[target_col])
    test_pred = final_model.predict(test_df[feature_cols])
    test_metrics = regression_metrics(test_df[target_col], test_pred)

    test_predictions = test_df[["date", "year"]].copy()
    test_predictions["model_name"] = model_name
    test_predictions["split"] = "test_2024"
    test_predictions["fold"] = np.nan
    test_predictions["actual_nd_daily_mwh"] = test_df[target_col].to_numpy()
    test_predictions["predicted_nd_daily_mwh"] = test_pred
    test_predictions["residual_mwh"] = (
        test_predictions["actual_nd_daily_mwh"] - test_predictions["predicted_nd_daily_mwh"]
    )

    summary = {
        "model_name": model_name,
        "feature_cols": ", ".join(feature_cols),
        "cv_mae_mean": cv_results["mae"].mean(),
        "cv_rmse_mean": cv_results["rmse"].mean(),
        "cv_wape_mean": cv_results["wape"].mean(),
        "cv_mbe_mean": cv_results["mbe"].mean(),
        "cv_r2_mean": cv_results["r2"].mean(),
        "test_mae": test_metrics["mae"],
        "test_rmse": test_metrics["rmse"],
        "test_wape": test_metrics["wape"],
        "test_mbe": test_metrics["mbe"],
        "test_r2": test_metrics["r2"],
    }

    return final_model, pd.DataFrame([summary]), cv_results, pd.concat([cv_predictions, test_predictions], ignore_index=True)

## 8. Baseline model — multiple linear regression

This preserves the colleague's interpretable baseline model.

In [ ]:
def make_linear_model() -> Pipeline:
    categorical_features = ["month", "day_of_week"]
    numeric_features = ["year", "is_holiday_gb_any", "hdd", "cdd"]

    preprocessor = make_tabular_preprocessor(categorical_features, numeric_features)

    return Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression()),
    ])


linear_model, linear_summary, linear_cv, linear_predictions = evaluate_sklearn_pipeline(
    model_name="linear_regression_base",
    model_factory=make_linear_model,
    df=model_df,
    dev_df=dev_df,
    test_df=test_df,
    folds=FOLDS,
    feature_cols=BASE_FEATURES,
)

display(linear_summary.T)
display(linear_cv)

## 9. GAM comparison

This keeps the GAM family tests from your aligned and expanded-feature notebooks.

If `pygam` is not installed, this section is skipped and the comparison table will continue with the remaining model families.

In [ ]:
def evaluate_gam_spec(
    model_name: str,
    feature_cols: list[str],
    builder: Callable[[], "LinearGAM"],
    dev_df: pd.DataFrame,
    test_df: pd.DataFrame,
    folds: list[dict],
    lam_grid: np.ndarray | None = None,
    target_col: str = TARGET_COL,
):
    if not HAS_PYGAM:
        print("Skipping GAM section because `pygam` is not installed.")
        return None, pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    if lam_grid is None:
        lam_grid = np.logspace(-3, 3, 7)

    cv_rows = []
    cv_preds = []

    for fold in folds:
        train_df, val_df, X_train_df, y_train, X_val_df, y_val = split_fold_data(
            df=dev_df,
            fold=fold,
            feature_cols=feature_cols,
            target_col=target_col,
        )

        gam = builder()
        gam.gridsearch(X_train_df.to_numpy(), y_train, lam=lam_grid, progress=False)
        pred_val = gam.predict(X_val_df.to_numpy())
        metrics = regression_metrics(y_val, pred_val)

        cv_rows.append({
            "model_name": model_name,
            "fold": fold["fold"],
            "train_start": fold["train_start"],
            "train_end": fold["train_end"],
            "val_start": fold["val_start"],
            "val_end": fold["val_end"],
            **metrics,
        })

        pred_df = val_df[["date", "year"]].copy()
        pred_df["model_name"] = model_name
        pred_df["split"] = "cv"
        pred_df["fold"] = fold["fold"]
        pred_df["actual_nd_daily_mwh"] = y_val
        pred_df["predicted_nd_daily_mwh"] = pred_val
        pred_df["residual_mwh"] = pred_df["actual_nd_daily_mwh"] - pred_df["predicted_nd_daily_mwh"]
        cv_preds.append(pred_df)

    cv_results = pd.DataFrame(cv_rows)
    cv_predictions = pd.concat(cv_preds, ignore_index=True) if cv_preds else pd.DataFrame()

    final_gam = builder()
    final_gam.gridsearch(dev_df[feature_cols].to_numpy(), dev_df[target_col].to_numpy(), lam=lam_grid, progress=False)
    test_pred = final_gam.predict(test_df[feature_cols].to_numpy())
    test_metrics = regression_metrics(test_df[target_col].to_numpy(), test_pred)

    test_predictions = test_df[["date", "year"]].copy()
    test_predictions["model_name"] = model_name
    test_predictions["split"] = "test_2024"
    test_predictions["fold"] = np.nan
    test_predictions["actual_nd_daily_mwh"] = test_df[target_col].to_numpy()
    test_predictions["predicted_nd_daily_mwh"] = test_pred
    test_predictions["residual_mwh"] = (
        test_predictions["actual_nd_daily_mwh"] - test_predictions["predicted_nd_daily_mwh"]
    )

    summary = pd.DataFrame([{
        "model_name": model_name,
        "feature_cols": ", ".join(feature_cols),
        "cv_mae_mean": cv_results["mae"].mean(),
        "cv_rmse_mean": cv_results["rmse"].mean(),
        "cv_wape_mean": cv_results["wape"].mean(),
        "cv_mbe_mean": cv_results["mbe"].mean(),
        "cv_r2_mean": cv_results["r2"].mean(),
        "test_mae": test_metrics["mae"],
        "test_rmse": test_metrics["rmse"],
        "test_wape": test_metrics["wape"],
        "test_mbe": test_metrics["mbe"],
        "test_r2": test_metrics["r2"],
    }])

    predictions = pd.concat([cv_predictions, test_predictions], ignore_index=True)
    return final_gam, summary, cv_results, predictions


def build_gam_base():
    return LinearGAM(
        f(0) +  # month
        f(1) +  # day_of_week
        f(2) +  # is_holiday_gb_any
        s(3, n_splines=20) +  # hdd
        s(4, n_splines=10)    # cdd
    )


def build_gam_year_smooth():
    return LinearGAM(
        s(0, n_splines=6) +   # year
        f(1) +                # month
        f(2) +                # day_of_week
        f(3) +                # is_holiday_gb_any
        s(4, n_splines=20) +  # hdd
        s(5, n_splines=10)    # cdd
    )


def build_gam_time_index_day_of_year():
    return LinearGAM(
        s(0, n_splines=12) +  # time_index
        s(1, n_splines=20) +  # day_of_year
        f(2) +                # day_of_week
        f(3) +                # is_holiday_gb_any
        s(4, n_splines=20) +  # hdd
        s(5, n_splines=10)    # cdd
    )


gam_specs = [
    ("gam_base", ["month", "day_of_week", "is_holiday_gb_any", "hdd", "cdd"], build_gam_base),
    ("gam_year_smooth", ["year", "month", "day_of_week", "is_holiday_gb_any", "hdd", "cdd"], build_gam_year_smooth),
    (
        "gam_time_index_day_of_year",
        ["time_index", "day_of_year", "day_of_week", "is_holiday_gb_any", "hdd", "cdd"],
        build_gam_time_index_day_of_year,
    ),
]

gam_models = {}
gam_summaries = []
gam_cv_results = []
gam_predictions = []

for model_name, feature_cols, builder in gam_specs:
    fitted, summary, cv_results, predictions = evaluate_gam_spec(
        model_name=model_name,
        feature_cols=feature_cols,
        builder=builder,
        dev_df=dev_df,
        test_df=test_df,
        folds=FOLDS,
    )
    if not summary.empty:
        gam_models[model_name] = fitted
        gam_summaries.append(summary)
        gam_cv_results.append(cv_results)
        gam_predictions.append(predictions)

gam_summary = pd.concat(gam_summaries, ignore_index=True) if gam_summaries else pd.DataFrame()
display(gam_summary.sort_values("test_rmse") if not gam_summary.empty else gam_summary)

## 10. Random forest comparison

This keeps the random-forest modelling family from your aligned comparison, but uses one clean pipeline and a single reduced grid. Expand the grid only when you deliberately want a slower tuning run.

In [ ]:
RF_FEATURES = BASE_FEATURES

RF_PARAM_GRID_SMALL = {
    "regressor__n_estimators": [300],
    "regressor__max_depth": [12, None],
    "regressor__min_samples_leaf": [1, 5],
    "regressor__max_features": ["sqrt"],
}

RUN_FULL_RF_GRID = False

RF_PARAM_GRID_FULL = {
    "regressor__n_estimators": [300, 600],
    "regressor__max_depth": [8, 12, 16, None],
    "regressor__min_samples_leaf": [1, 5, 10],
    "regressor__max_features": ["sqrt", 0.5, None],
}

RF_PARAM_GRID = RF_PARAM_GRID_FULL if RUN_FULL_RF_GRID else RF_PARAM_GRID_SMALL


def make_rf_pipeline(params: dict | None = None) -> Pipeline:
    categorical_features = ["month", "day_of_week", "is_holiday_gb_any"]
    numeric_features = ["year", "hdd", "cdd"]

    preprocessor = make_tabular_preprocessor(categorical_features, numeric_features)

    regressor = RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", regressor),
    ])

    if params:
        model.set_params(**params)

    return model


def tune_rf_on_folds(
    dev_df: pd.DataFrame,
    folds: list[dict],
    feature_cols: list[str],
    param_grid: dict,
    target_col: str = TARGET_COL,
) -> pd.DataFrame:
    rows = []

    for params in ParameterGrid(param_grid):
        fold_metrics = []

        for fold in folds:
            train_df, val_df, X_train, y_train, X_val, y_val = split_fold_data(
                df=dev_df,
                fold=fold,
                feature_cols=feature_cols,
                target_col=target_col,
            )

            model = make_rf_pipeline(params)
            model.fit(X_train, y_train)
            pred_val = model.predict(X_val)
            fold_metrics.append(regression_metrics(y_val, pred_val))

        rows.append({
            **params,
            "cv_mae_mean": np.mean([m["mae"] for m in fold_metrics]),
            "cv_rmse_mean": np.mean([m["rmse"] for m in fold_metrics]),
            "cv_wape_mean": np.mean([m["wape"] for m in fold_metrics]),
            "cv_r2_mean": np.mean([m["r2"] for m in fold_metrics]),
        })

    return pd.DataFrame(rows).sort_values(["cv_mae_mean", "cv_rmse_mean"]).reset_index(drop=True)


rf_tuning_results = tune_rf_on_folds(
    dev_df=dev_df,
    folds=FOLDS,
    feature_cols=RF_FEATURES,
    param_grid=RF_PARAM_GRID,
)

display(rf_tuning_results)

rf_best_params = {
    key: rf_tuning_results.iloc[0][key]
    for key in RF_PARAM_GRID.keys()
}

# Convert float-looking values back where needed after dataframe round-tripping.
if pd.notna(rf_best_params["regressor__max_depth"]):
    rf_best_params["regressor__max_depth"] = int(rf_best_params["regressor__max_depth"])
else:
    rf_best_params["regressor__max_depth"] = None

rf_best_params["regressor__n_estimators"] = int(rf_best_params["regressor__n_estimators"])
rf_best_params["regressor__min_samples_leaf"] = int(rf_best_params["regressor__min_samples_leaf"])

print("Selected RF params:", rf_best_params)


rf_model, rf_summary, rf_cv, rf_predictions = evaluate_sklearn_pipeline(
    model_name="random_forest_base_tuned",
    model_factory=lambda: make_rf_pipeline(rf_best_params),
    df=model_df,
    dev_df=dev_df,
    test_df=test_df,
    folds=FOLDS,
    feature_cols=RF_FEATURES,
)

display(rf_summary.T)

## 11. XGBoost expanded feature comparison

This preserves the colleague's final XGBoost testing workflow.

The three variants test whether temporal features improve out-of-sample performance:

1. `time_index` with month/day/holiday/HDD/CDD;
2. `year + day_of_year` with month/day/holiday/HDD/CDD;
3. `time_index + day_of_year` with month/day/holiday/HDD/CDD.

The third variant was previously selected as the final specification.

In [ ]:
if not HAS_XGBOOST:
    raise ImportError("xgboost is required for the final selected model family. Install it before running this section.")


def prepare_xgb_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["month"] = out["month"].astype("category")
    out["day_of_week"] = out["day_of_week"].astype("category")
    out["is_holiday_gb_any"] = out["is_holiday_gb_any"].astype(int)
    return out


def make_xgb_regressor() -> "XGBRegressor":
    return XGBRegressor(
        objective="reg:squarederror",
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        enable_categorical=True,
        tree_method="hist",
    )


xgb_model_variants = {
    "xgb_model_1_time_index": EXPANDED_FEATURES_TIME_INDEX,
    "xgb_model_2_year_day_of_year": EXPANDED_FEATURES_YEAR_DOY,
    "xgb_model_3_time_index_day_of_year": EXPANDED_FEATURES_TIME_INDEX_DOY,
}


def evaluate_xgb_variant(
    model_name: str,
    feature_cols: list[str],
    dev_df: pd.DataFrame,
    test_df: pd.DataFrame,
    folds: list[dict],
    target_col: str = TARGET_COL,
):
    xgb_dev_df = prepare_xgb_frame(dev_df)
    xgb_test_df = prepare_xgb_frame(test_df)

    cv_rows = []
    cv_preds = []

    for fold in folds:
        train_df, val_df, X_train, y_train, X_val, y_val = split_fold_data(
            df=xgb_dev_df,
            fold=fold,
            feature_cols=feature_cols,
            target_col=target_col,
        )

        X_train = prepare_xgb_frame(X_train)
        X_val = prepare_xgb_frame(X_val)

        model = make_xgb_regressor()
        model.fit(X_train, y_train)
        pred_val = model.predict(X_val)
        metrics = regression_metrics(y_val, pred_val)

        cv_rows.append({
            "model_name": model_name,
            "fold": fold["fold"],
            "train_start": fold["train_start"],
            "train_end": fold["train_end"],
            "val_start": fold["val_start"],
            "val_end": fold["val_end"],
            **metrics,
        })

        pred_df = val_df[["date", "year"]].copy()
        pred_df["model_name"] = model_name
        pred_df["split"] = "cv"
        pred_df["fold"] = fold["fold"]
        pred_df["actual_nd_daily_mwh"] = y_val
        pred_df["predicted_nd_daily_mwh"] = pred_val
        pred_df["residual_mwh"] = pred_df["actual_nd_daily_mwh"] - pred_df["predicted_nd_daily_mwh"]
        cv_preds.append(pred_df)

    cv_results = pd.DataFrame(cv_rows)
    cv_predictions = pd.concat(cv_preds, ignore_index=True)

    final_model = make_xgb_regressor()
    final_model.fit(xgb_dev_df[feature_cols], xgb_dev_df[target_col])

    test_pred = final_model.predict(xgb_test_df[feature_cols])
    test_metrics = regression_metrics(xgb_test_df[target_col], test_pred)

    test_predictions = xgb_test_df[["date", "year"]].copy()
    test_predictions["model_name"] = model_name
    test_predictions["split"] = "test_2024"
    test_predictions["fold"] = np.nan
    test_predictions["actual_nd_daily_mwh"] = xgb_test_df[target_col].to_numpy()
    test_predictions["predicted_nd_daily_mwh"] = test_pred
    test_predictions["residual_mwh"] = (
        test_predictions["actual_nd_daily_mwh"] - test_predictions["predicted_nd_daily_mwh"]
    )

    feature_importance = pd.DataFrame({
        "model_name": model_name,
        "feature": feature_cols,
        "importance": final_model.feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    summary = pd.DataFrame([{
        "model_name": model_name,
        "feature_cols": ", ".join(feature_cols),
        "cv_mae_mean": cv_results["mae"].mean(),
        "cv_rmse_mean": cv_results["rmse"].mean(),
        "cv_wape_mean": cv_results["wape"].mean(),
        "cv_mbe_mean": cv_results["mbe"].mean(),
        "cv_r2_mean": cv_results["r2"].mean(),
        "test_mae": test_metrics["mae"],
        "test_rmse": test_metrics["rmse"],
        "test_wape": test_metrics["wape"],
        "test_mbe": test_metrics["mbe"],
        "test_r2": test_metrics["r2"],
    }])

    predictions = pd.concat([cv_predictions, test_predictions], ignore_index=True)
    return final_model, summary, cv_results, predictions, feature_importance


xgb_models = {}
xgb_summaries = []
xgb_cv_results = []
xgb_predictions = []
xgb_feature_importances = []

for model_name, feature_cols in xgb_model_variants.items():
    print("=" * 80)
    print("Running", model_name)
    fitted, summary, cv_results, predictions, feature_importance = evaluate_xgb_variant(
        model_name=model_name,
        feature_cols=feature_cols,
        dev_df=dev_df,
        test_df=test_df,
        folds=FOLDS,
    )
    xgb_models[model_name] = fitted
    xgb_summaries.append(summary)
    xgb_cv_results.append(cv_results)
    xgb_predictions.append(predictions)
    xgb_feature_importances.append(feature_importance)

xgb_summary = pd.concat(xgb_summaries, ignore_index=True)
xgb_cv_results_all = pd.concat(xgb_cv_results, ignore_index=True)
xgb_predictions_all = pd.concat(xgb_predictions, ignore_index=True)
xgb_feature_importance_all = pd.concat(xgb_feature_importances, ignore_index=True)

display(xgb_summary.sort_values("test_rmse").reset_index(drop=True))
display(xgb_feature_importance_all)

## 12. Overall model-family comparison

This is the key integration output: it shows why the final model was selected, while preserving the evidence trail from the alternative modelling families.

In [ ]:
summary_tables = [
    linear_summary,
    rf_summary,
    xgb_summary,
]

if not gam_summary.empty:
    summary_tables.insert(1, gam_summary)

model_comparison_df = pd.concat(summary_tables, ignore_index=True)

metric_cols = [
    "cv_mae_mean", "cv_rmse_mean", "cv_wape_mean", "cv_mbe_mean", "cv_r2_mean",
    "test_mae", "test_rmse", "test_wape", "test_mbe", "test_r2",
]

model_comparison_display = model_comparison_df.sort_values("test_rmse").reset_index(drop=True)
display(model_comparison_display[["model_name", *metric_cols, "feature_cols"]])

## 13. Final model decision

The expected selected model is:

`xgb_model_3_time_index_day_of_year`

This specification combines:

- `time_index` for long-run trend/drift;
- `day_of_year` for within-year seasonal position;
- `month` and `day_of_week` for calendar structure;
- GB holiday flag;
- `hdd` and `cdd` for climate responsiveness.

After reviewing the comparison table above, only override this if rerun results clearly show another model has materially better out-of-sample performance and the change is documented in the PR.

In [ ]:
SELECTED_MODEL_NAME = "xgb_model_3_time_index_day_of_year"

selected_row = model_comparison_df.loc[
    model_comparison_df["model_name"] == SELECTED_MODEL_NAME
].copy()

if selected_row.empty:
    raise ValueError(f"Selected model `{SELECTED_MODEL_NAME}` was not found in comparison results.")

display(selected_row.T)

selected_feature_cols = xgb_model_variants[SELECTED_MODEL_NAME]
selected_model_dev_fit = xgb_models[SELECTED_MODEL_NAME]

selected_predictions_2024 = xgb_predictions_all[
    (xgb_predictions_all["model_name"] == SELECTED_MODEL_NAME)
    & (xgb_predictions_all["split"] == "test_2024")
].copy()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(selected_predictions_2024["date"], selected_predictions_2024["actual_nd_daily_mwh"], label="Actual")
ax.plot(selected_predictions_2024["date"], selected_predictions_2024["predicted_nd_daily_mwh"], label="Predicted", linestyle="--")
ax.set_title(f"{SELECTED_MODEL_NAME}: actual vs predicted daily ND demand, 2024 holdout")
ax.set_xlabel("Date")
ax.set_ylabel("ND daily demand (MWh)")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(selected_predictions_2024["residual_mwh"], bins=40)
ax.set_title(f"{SELECTED_MODEL_NAME}: residual distribution, 2024 holdout")
ax.set_xlabel("Residual MWh: actual - predicted")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 14. Refit selected model on all historic data

Use the development/test comparison above for model selection. Once the model choice is locked, refit the selected model on all available historic data before using it for future climate-member prediction.

This cell creates the fitted object in memory. Saving model artefacts is disabled by default because `.joblib`/`.pkl` files should not be committed.

In [ ]:
full_history_df = prepare_xgb_frame(model_df)
final_selected_model = make_xgb_regressor()
final_selected_model.fit(full_history_df[selected_feature_cols], full_history_df[TARGET_COL])

print("Final selected model fitted on full historic model data.")
print("Selected features:", selected_feature_cols)
print("Training rows:", len(full_history_df))

## 15. Optional local outputs and validation evidence

By default, this notebook does not write files. When preparing a PR, the useful small validation evidence is usually:

- `demand_model_comparison_metrics.csv`
- `demand_model_cv_results.csv`
- `demand_model_selection_README.md`

Large generated outputs and model artefacts should remain local.

In [ ]:
all_cv_tables = [linear_cv, rf_cv, xgb_cv_results_all]
if gam_cv_results:
    all_cv_tables.insert(1, pd.concat(gam_cv_results, ignore_index=True))

cv_results_all_df = pd.concat(all_cv_tables, ignore_index=True)

all_prediction_tables = [linear_predictions, rf_predictions, xgb_predictions_all]
if gam_predictions:
    all_prediction_tables.insert(1, pd.concat(gam_predictions, ignore_index=True))

predictions_all_df = pd.concat(all_prediction_tables, ignore_index=True)

selection_readme = f"""# Objective 2 Task 3 demand model selection

## Purpose
This document records the integrated daily demand model comparison for Objective 2 Task 3.

## Integrated source notebooks
- `merge.ipynb`
- `EDA.ipynb`
- `gam_rf_pri_aligned.ipynb`
- `gam_year_smoothing_additional_feature_testing.ipynb`
- `Objective 2 Task 3.ipynb`

## Target
- `nd_daily_mwh`

## Validation setup
- Development period: years before {TEST_YEAR}
- Holdout test year: {TEST_YEAR}
- Cross-validation: expanding-window year folds over 2019-2023
- Metrics: MAE, RMSE, WAPE, MBE, R²

## Model families compared
- Multiple linear regression
- GAM variants
- Random forest
- XGBoost expanded feature variants

## Selected model
`{SELECTED_MODEL_NAME}`

Selected features:
{selected_feature_cols}

## Notes
The selected XGBoost variant retains climate responsiveness through `hdd` and `cdd`, while adding temporal features for long-run drift and seasonal position.
"""

if WRITE_VALIDATION_EVIDENCE:
    VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
    model_comparison_df.to_csv(VALIDATION_DIR / "demand_model_comparison_metrics.csv", index=False)
    cv_results_all_df.to_csv(VALIDATION_DIR / "demand_model_cv_results.csv", index=False)
    (VALIDATION_DIR / "demand_model_selection_README.md").write_text(selection_readme, encoding="utf-8")
    print("Wrote validation evidence to", VALIDATION_DIR)

if WRITE_LOCAL_OUTPUTS:
    LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    predictions_all_df.to_csv(LOCAL_OUTPUT_DIR / "demand_model_predictions_all.csv", index=False)
    xgb_feature_importance_all.to_csv(LOCAL_OUTPUT_DIR / "xgb_feature_importance.csv", index=False)
    print("Wrote local outputs to", LOCAL_OUTPUT_DIR)

print(selection_readme)

## 16. Handoff to future-demand generation

The selected daily model should be used downstream to estimate daily climate response for UKCP18 members. The downstream Objective 2 workflow should then:

1. predict climate-sensitive daily demand for each climate member;
2. anchor/scalably adjust annual totals to FES ED1 values for `Holistic Transition` and `Electric Engagement`;
3. disaggregate daily demand to hourly demand using the historic hourly shape library;
4. output `demand_future_hourly_2030_2045` with:
   - `timestamp_utc`
   - `year`
   - `fes_scenario`
   - `climate_member`
   - `demand_mw`